<div style="display:flex;align-items:center;justify-content:space-between;border-bottom:2px solid #c8962d;padding-bottom:12px;margin-bottom:20px">
  <div><strong>Universidad Externado de Colombia</strong><br>
  <span>Programa de Ciencia de Datos · Machine Learning II</span><br>
  <span>Docente: Wilmer Pineda-Ríos</span></div>
  <img src="../../assets/brand/logo-externado.png" width="190">
</div>

# Taller evaluativo — Bosques, importancia y boosting

**Duración:** 120 minutos · **Modalidad:** parejas · **Valor:** 1,0 % del curso

In [ ]:
INTEGRANTE_1 = {'nombre': 'Santiago Sandoval', 'codigo': '1019763993'}
INTEGRANTE_2 = {'nombre': 'Miguel Moreno', 'codigo': '1021497559'}

## Instrucciones

Complete únicamente las secciones marcadas. No cambie la partición, los folds, la semilla ni los modelos. Test permanece cerrado hasta la sección 6.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
pd.set_option("display.max_rows", 30)
pd.set_option("display.precision", 4)

In [3]:
candidates = [
    Path("../../datasets/public/Customer_Churn_Iran.csv"),
    Path("datasets/public/Customer_Churn_Iran.csv"),
    Path("../datasets/public/Customer_Churn_Iran.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("No se encontró Customer_Churn_Iran.csv")

df = pd.read_csv(data_path)
target = "Churn"
X = df.drop(columns=target)
y = df[target].astype(int)

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "balanced_accuracy": "balanced_accuracy",
    "recall": "recall",
    "precision": "precision",
}

pd.DataFrame({
    "filas": [len(df)],
    "predictores": [X.shape[1]],
    "churn_n": [int(y.sum())],
    "churn_pct": [100 * y.mean()],
})

,filas,predictores,churn_n,churn_pct
0,3150,13,495,15.7143


## 1. Auditoría

Reporte tamaño, faltantes y proporción de churn. Explique en dos líneas por qué accuracy no será la métrica principal.

In [4]:
auditoria = pd.DataFrame({
    'filas': [len(df)],
    'predictores': [X.shape[1]],
    'faltantes': [df.isnull().sum().sum()],
    'churn_n': [int(y.sum())],
    'churn_pct': [100 * y.mean()]
})
auditoria

,filas,predictores,faltantes,churn_n,churn_pct
0,3150,13,0,495,15.7143


**Justificación de la métrica de evaluación:**

Dado el desbalance de clases (el churn representa solo el 15.71% de la muestra), la métrica de *accuracy* resulta engañosa: un modelo ingenuo que clasifique a todos los clientes como no-churn obtendría un 84.29% de exactitud sin detectar ninguna fuga. Por ello, se evalúa mediante *balanced accuracy*, *recall* y *precision*, enfocadas en el desempeño sobre la clase de interés.

## 2. Modelos prescritos

In [5]:
tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=15, random_state=RANDOM_STATE)
forest = RandomForestClassifier(
    n_estimators=300, max_features="sqrt", min_samples_leaf=5,
    bootstrap=True, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1,
)
adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    n_estimators=200, learning_rate=0.5, random_state=RANDOM_STATE,
)
models = {"Árbol": tree, "Random Forest": forest, "AdaBoost": adaboost}

## 3. Validación cruzada

Complete el ciclo y reporte media y desviación estándar de las tres métricas.

In [6]:
rows = []
for name, model in models.items():
    res = cross_validate(model, X_dev, y_dev, cv=cv, scoring=scoring)
    rows.append({
        "modelo": name,
        "balanced_accuracy_mean": res["test_balanced_accuracy"].mean(),
        "balanced_accuracy_std": res["test_balanced_accuracy"].std(),
        "recall_mean": res["test_recall"].mean(),
        "recall_std": res["test_recall"].std(),
        "precision_mean": res["test_precision"].mean(),
        "precision_std": res["test_precision"].std(),
    })
cv_results = pd.DataFrame(rows)
cv_results

,modelo,balanced_accuracy_mean,balanced_accuracy_std,recall_mean,recall_std,precision_mean,precision_std
0,Árbol,0.8120,0.0257,0.6592,0.0534,0.7773,0.0238
1,Random Forest,0.8782,0.0059,0.7752,0.0101,0.8848,0.0152
2,AdaBoost,0.7310,0.0318,0.4799,0.0727,0.8460,0.0658


**Modelo seleccionado y evidencia de CV:**

Se selecciona **Random Forest**.

**Evidencia de validación cruzada:**
1. **Balanced Accuracy:** Random Forest alcanza el valor medio más alto (0.8782 vs. 0.8120 del Árbol y 0.7310 de AdaBoost).
2. **Recall:** Logra la mayor capacidad de detección de clientes en riesgo con un recall medio de 0.7752 (vs. 0.6592 del Árbol y 0.4799 de AdaBoost).
3. **Precision:** Presenta la precisión más elevada con 0.8848 (vs. 0.7773 del Árbol y 0.8460 de AdaBoost).
4. **Estabilidad:** Ofrece la menor desviación estándar entre folds en todas las métricas (std de balanced_accuracy = 0.0059), demostrando gran consistencia predictiva sin mirada en test.

## 4. Estimación OOB

Ajuste Random Forest sobre desarrollo. Compare `oob_score_` con `balanced_accuracy` calculada desde `oob_decision_function_`.

In [7]:
forest.fit(X_dev, y_dev)
oob_acc = forest.oob_score_
oob_probs = forest.oob_decision_function_
oob_preds = np.argmax(oob_probs, axis=1)
oob_bal_acc = balanced_accuracy_score(y_dev, oob_preds)

oob_results = pd.Series({
    "oob_score_ (Accuracy)": oob_acc,
    "balanced_accuracy (OOB)": oob_bal_acc
}, name="resultado")
oob_results

oob_score_ (Accuracy)      0.9520
balanced_accuracy (OOB)    0.8904
Name: resultado, dtype: float64

**Comparación de estimaciones OOB:**

El atributo `oob_score_` arroja una exactitud global (Accuracy) del **95.20%**, la cual está distorsionada por la alta proporción de la clase no-churn. En contraste, la `balanced_accuracy` calculada a partir de `oob_decision_function_` es del **89.04%**, reflejando de manera equilibrada y justa la capacidad predictiva sobre ambas clases sin sobreestimar el rendimiento.

## 5. Importancias

Construya tablas con las diez primeras variables por MDI y por permutación. Use el primer fold de validación y 20 repeticiones.

In [8]:
mdi_importances = forest.feature_importances_
mdi_top10 = pd.DataFrame({
    "variable": X.columns,
    "importancia_mdi": mdi_importances
}).sort_values("importancia_mdi", ascending=False).head(10).reset_index(drop=True)
mdi_top10

,variable,importancia_mdi
0,Complains,0.2317
1,Status,0.1397
2,Seconds of Use,0.1167
3,Frequency of use,0.1099
4,Subscription Length,0.0930
5,Distinct Called Numbers,0.0758
6,Customer Value,0.0716
7,Call Failure,0.0531
8,Frequency of SMS,0.0447
9,Age,0.0249


In [9]:
train_idx, val_idx = next(cv.split(X_dev, y_dev))
X_train_f1, y_train_f1 = X_dev.iloc[train_idx], y_dev.iloc[train_idx]
X_val_f1, y_val_f1 = X_dev.iloc[val_idx], y_dev.iloc[val_idx]

rf_f1 = RandomForestClassifier(
    n_estimators=300, max_features="sqrt", min_samples_leaf=5,
    bootstrap=True, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1,
)
rf_f1.fit(X_train_f1, y_train_f1)

perm_res = permutation_importance(
    rf_f1, X_val_f1, y_val_f1, n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1
)
permutation_top10 = pd.DataFrame({
    "variable": X.columns,
    "importancia_permutacion_media": perm_res.importances_mean,
    "importancia_permutacion_std": perm_res.importances_std
}).sort_values("importancia_permutacion_media", ascending=False).head(10).reset_index(drop=True)
permutation_top10

,variable,importancia_permutacion_media,importancia_permutacion_std
0,Complains,0.0554,0.0086
1,Status,0.0263,0.0082
2,Distinct Called Numbers,0.0215,0.0056
3,Frequency of use,0.0183,0.0041
4,Seconds of Use,0.0165,0.0046
5,Subscription Length,0.0123,0.0041
6,Customer Value,0.0070,0.0027
7,Frequency of SMS,0.0033,0.0029
8,Call Failure,0.0012,0.0040
9,Tariff Plan,0.0000,0.0000


**Coincidencia, diferencia y limitación de los rankings:**

1. **Coincidencia:** Ambos métodos coinciden en señalar a **`Complains`** (quejas) y **`Status`** (estado activo/inactivo) como las dos variables predictivas principales del churn.
2. **Diferencia:** **`Distinct Called Numbers`** sube al 3.er puesto en permutación sobre validación (caída media de 0.0215 en precisión), mientras que en MDI ocupaba el 6.º lugar. Asimismo, **`Call Failure`** pasa del puesto 8 en MDI al puesto 9 en permutación.
3. **Limitación (Cardinalidad / Correlación):** MDI sobreestima predictores continuos o de alta cardinalidad (como `Seconds of Use` y `Customer Value`) por disponer de múltiples puntos de corte en el entrenamiento. En contraste, la permutación evalúa el impacto real fuera de muestra, pero puede subestimar predictores colineales (como `Frequency of use` y `Seconds of Use`), ya que el modelo se apoya en la variable correlacionada no permutada para mantener sus predicciones.

## 6. Test: una sola consulta

Ajuste el modelo ya seleccionado con todo desarrollo. Reporte las cuatro métricas y la matriz de confusión.

In [10]:
rf_final = RandomForestClassifier(
    n_estimators=300, max_features="sqrt", min_samples_leaf=5,
    bootstrap=True, oob_score=True, random_state=RANDOM_STATE, n_jobs=-1,
)
rf_final.fit(X_dev, y_dev)
y_pred_test = rf_final.predict(X_test)

final_metrics = pd.Series({
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred_test),
    "accuracy": accuracy_score(y_test, y_pred_test),
    "precision": precision_score(y_test, y_pred_test),
    "recall": recall_score(y_test, y_pred_test),
}, name="resultado")

final_confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred_test),
    index=["Actual 0", "Actual 1"],
    columns=["Pred 0", "Pred 1"]
)
final_metrics, final_confusion

,Pred 0,Pred 1
Actual 0,521,10
Actual 1,14,85


### Verificación de cálculos manuales a partir de la Matriz de Confusión

A partir de la matriz de confusión obtenida en test ($TN=521$, $FP=10$, $FN=14$, $TP=85$):

1. **Accuracy:** $\frac{TN + TP}{TN + FP + FN + TP} = \frac{521 + 85}{521 + 10 + 14 + 85} = \frac{606}{630} = 0.9619$ ($96.19\%$)
2. **Precision:** $\frac{TP}{TP + FP} = \frac{85}{85 + 10} = \frac{85}{95} = 0.8947$ ($89.47\%$)
3. **Recall (Sensibilidad):** $\frac{TP}{TP + FN} = \frac{85}{85 + 14} = \frac{85}{99} = 0.8586$ ($85.86\%$)
4. **Especificidad (Sensibilidad Clase 0):** $\frac{TN}{TN + FP} = \frac{521}{521 + 10} = \frac{521}{531} = 0.9812$ ($98.12\%$)
5. **Balanced Accuracy:** $\frac{\text{Recall} + \text{Especificidad}}{2} = \frac{0.8586 + 0.9812}{2} = 0.9199$ ($91.99\%$)

## 7. Recomendación

Máximo ocho líneas: modelo, evidencia, error relevante, utilidad operativa y advertencia sobre causalidad.

> Se recomienda implementar el modelo **Random Forest** (300 árboles, max_features='sqrt', min_samples_leaf=5), respaldado por un *balanced accuracy* de 0.8782 en validación cruzada y 0.9199 en test (*accuracy*=0.9619, *precision*=0.8947, *recall*=0.8586).
> El principal riesgo operativo se concentra en los 14 falsos negativos (14.14% de clientes fugados no detectados), lo que representa pérdida directa de ingresos.
> Operativamente, el modelo permite enfocar campañas de retención prioritarias en clientes con quejas registradas (`Complains`) o estado inactivo (`Status`).
> **Advertencia explícita:** La importancia predictiva identificada por MDI o permutación no demuestra causalidad; modificar una variable no garantiza una reducción del churn sin una prueba controlada.

## Lista de entrega

- [x] Identidad completa.
- [x] Notebook ejecutado.
- [x] Mismos folds para los tres modelos.
- [x] Test consultado una vez.
- [x] Dos rankings de importancia.
- [x] Recomendación sin afirmaciones causales.